**Training data: The 707 competition train images only.**

In [1]:
!pip install -q ultralytics
!pip install -q segmentation-models-pytorch
!pip install -q albumentations

In [2]:
import cv2
import csv
import os
import glob
import torch
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from pycocotools import mask as mask_utils
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEST_IMAGES_DIR = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images"

class Config_YOLO26x():
    MODEL_PATH = "/kaggle/input/models/hdjojo/yolo26x-seg-solar-filament/pytorch/default/1/yolo26x-seg-solar-filament-2048-1109.pt"
    IMG_SIZE = 2048
    CONF = 0.25
    IOU = 0.0
    MAX_DET = 100
    
CONFIG_YOLO26x = Config_YOLO26x()
model_yolo26x = YOLO(CONFIG_YOLO26x.MODEL_PATH)

class Config_YOLOv8x():
    MODEL_PATH = "/kaggle/input/models/hdjojo/yolov8x-seg-solar-filament/pytorch/default/1/yolov8x-seg-solar-filament-2048-1110.pt"
    IMG_SIZE = 2048
    CONF = 0.25
    IOU = 0.0
    MAX_DET = 100
    
CONFIG_YOLOv8x = Config_YOLOv8x()
model_yolov8x = YOLO(CONFIG_YOLOv8x.MODEL_PATH)

class Config_UNetEffNetB5():
    MODEL_PATH = "/kaggle/input/models/hdjojo/smp-unet-solar-filament/pytorch/default/1/smp-unet-effnet-b5-solar-filament-1536-1114.pth"
    ENCODER_NAME = "timm-efficientnet-b5"
    IMG_SIZE = 1536

CONFIG_UNetEffNetB5 = Config_UNetEffNetB5()
model_uneteffnetb5 = smp.Unet(encoder_name=CONFIG_UNetEffNetB5.ENCODER_NAME, encoder_weights=None, in_channels=1, classes=1, activation=None)
model_uneteffnetb5.to(DEVICE)
model_uneteffnetb5.load_state_dict(torch.load(CONFIG_UNetEffNetB5.MODEL_PATH))
model_uneteffnetb5.eval()

class Config_UNetEffNetB4():
    MODEL_PATH = "/kaggle/input/models/hdjojo/smp-unet-solar-filament/pytorch/default/1/smp-unet-effnet-b4-solar-filament-1856-1115.pth"
    ENCODER_NAME = "timm-efficientnet-b4"
    IMG_SIZE = 1856

CONFIG_UNetEffNetB4 = Config_UNetEffNetB4()
model_uneteffnetb4 = smp.Unet(encoder_name=CONFIG_UNetEffNetB4.ENCODER_NAME, encoder_weights=None, in_channels=1, classes=1, activation=None)
model_uneteffnetb4.to(DEVICE)
model_uneteffnetb4.load_state_dict(torch.load(CONFIG_UNetEffNetB4.MODEL_PATH))
model_uneteffnetb4.eval()

class Config_UNetEffNetB3():
    MODEL_PATH = "/kaggle/input/models/hdjojo/smp-unet-solar-filament/pytorch/default/1/smp-unet-effnet-b3-solar-filament-2048-1114.pth"
    ENCODER_NAME = "timm-efficientnet-b3"
    IMG_SIZE = 2048

CONFIG_UNetEffNetB3 = Config_UNetEffNetB3()
model_uneteffnetb3 = smp.Unet(encoder_name=CONFIG_UNetEffNetB3.ENCODER_NAME, encoder_weights=None, in_channels=1, classes=1, activation=None)
model_uneteffnetb3.to(DEVICE)
model_uneteffnetb3.load_state_dict(torch.load(CONFIG_UNetEffNetB3.MODEL_PATH))
model_uneteffnetb3.eval()

class Config_UNetPPEffNetB2():
    MODEL_PATH = "/kaggle/input/models/hdjojo/smp-unet-solar-filament/pytorch/default/1/smp-unetplusplus-effnet-b2-solar-filament-2048-1115.pth"
    ENCODER_NAME = "timm-efficientnet-b2"
    IMG_SIZE = 2048

CONFIG_UNetPPEffNetB2 = Config_UNetPPEffNetB2()
model_unetppeffnetb2 = smp.UnetPlusPlus(encoder_name=CONFIG_UNetPPEffNetB2.ENCODER_NAME, encoder_weights=None, in_channels=1, classes=1, activation=None)
model_unetppeffnetb2.to(DEVICE)
model_unetppeffnetb2.load_state_dict(torch.load(CONFIG_UNetPPEffNetB2.MODEL_PATH))
model_unetppeffnetb2.eval()

UnetPlusPlus(
  (encoder): EfficientNetEncoder(
    (conv_stem): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): Swish()
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): Swish()
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): Swish()
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv_pw): Conv2d(32, 16, kernel_size=

In [4]:
def mask_to_rle(binary_mask):
    rle = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle["counts"]

def remove_instance_overlaps(masks, scores):
    
    sorted_indices = np.argsort(scores)[::-1]
    sorted_masks = masks[sorted_indices]
    
    accumulated_mask = np.zeros(masks.shape[1:], dtype=np.uint8)
    non_overlapping_masks = []
    
    for mask in sorted_masks:
        clean_mask = mask & ~accumulated_mask
        non_overlapping_masks.append(clean_mask)
        accumulated_mask |= mask
        
    return np.array(non_overlapping_masks)

def mask_iou(mask1: np.ndarray, mask2: np.ndarray) -> float:
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    if union == 0:
        return 0.0
    return intersection / union

def non_max_suppression_masks(masks: list, confidences: list, iou_threshold: float = 0.5) -> tuple:
    sorted_idx = np.argsort(confidences)[::-1]
    sorted_masks = [masks[i] for i in sorted_idx]
    sorted_confs = [confidences[i] for i in sorted_idx]

    keep_masks = []
    keep_confs = []

    while sorted_masks:
        current_mask = sorted_masks.pop(0)
        current_conf = sorted_confs.pop(0)

        keep_masks.append(current_mask)
        keep_confs.append(current_conf)

        remaining_masks = []
        remaining_confs = []
        for m, c in zip(sorted_masks, sorted_confs):
            if mask_iou(current_mask, m) <= iou_threshold:
                remaining_masks.append(m)
                remaining_confs.append(c)

        sorted_masks = remaining_masks
        sorted_confs = remaining_confs

    return np.array(keep_masks), np.array(keep_confs)

def accumulate_masks(masks, scores):
    sorted_indices = np.argsort(scores)[::-1]
    sorted_masks = masks[sorted_indices]
    
    accumulated_mask = np.zeros(masks.shape[1:], dtype=np.uint8)
    
    for mask in sorted_masks:
        clean_mask = mask & ~accumulated_mask
        accumulated_mask |= mask
        
    return accumulated_mask

def image_transform(img_size: int) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2(),
    ])
    
@torch.no_grad()
def predict_with_tta(model: nn.Module, image_gray: np.ndarray, image_size: int, device: str) -> np.ndarray:
    h, w = image_gray.shape[:2]
    img_trans = image_transform(image_size)

    def _infer(img_aug: np.ndarray) -> np.ndarray:
        inputs = img_trans(image=img_aug)['image'].unsqueeze(0).to(device)
        preds = torch.sigmoid(model(inputs)).squeeze().cpu().numpy()
        return cv2.resize(preds, (w, h), interpolation=cv2.INTER_LINEAR)

    p0 = _infer(image_gray)
    p1 = _infer(image_gray[:, ::-1].copy())
    p2 = _infer(image_gray[::-1, :].copy())
    p3 = _infer(image_gray[::-1, ::-1].copy())

    p1 = p1[:, ::-1]
    p2 = p2[::-1, :]
    p3 = p3[::-1, ::-1]

    return np.mean(np.stack([p0, p1, p2, p3], axis=0), axis=0).astype(np.float32)
    
def refine_mask(mask: np.ndarray, min_area: int = 200, close_kernel: int = 5,) -> np.ndarray:

    mask_u8 = mask.astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel, close_kernel))
    closed = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, kernel)

    n_lbl, labels, stats, _ = cv2.connectedComponentsWithStats(closed, connectivity=8)
    refined = np.zeros_like(closed)
    for lbl in range(1, n_lbl):
        if stats[lbl, cv2.CC_STAT_AREA] >= min_area:
            refined[labels == lbl] = 1
    
    return (refined > 0).astype(np.uint8)

In [5]:
submission_rows = []
for img_path in tqdm(sorted(glob.glob(f"{TEST_IMAGES_DIR}/*.jpeg")), desc="Inference"):
    file_stem = os.path.splitext(os.path.basename(img_path))[0]

    result_yolo26x = model_yolo26x.predict(
        img_path, imgsz=CONFIG_YOLO26x.IMG_SIZE, conf=CONFIG_YOLO26x.CONF, iou=CONFIG_YOLO26x.IOU, max_det=CONFIG_YOLO26x.MAX_DET, verbose=False)[0]
    
    result_yolov8x = model_yolov8x.predict(
        img_path, imgsz=CONFIG_YOLOv8x.IMG_SIZE, conf=CONFIG_YOLOv8x.CONF, iou=CONFIG_YOLOv8x.IOU, max_det=CONFIG_YOLOv8x.MAX_DET, verbose=False)[0]

    if result_yolo26x.masks is None or result_yolov8x.masks is None: continue

    result_yolo26x_masks_data = result_yolo26x.masks.data.cpu().numpy()
    scores_yolo26x = result_yolo26x.boxes.conf.cpu().numpy()

    result_yolov8x_masks_data = result_yolov8x.masks.data.cpu().numpy()
    scores_yolov8x = result_yolov8x.boxes.conf.cpu().numpy()

    result_combined_masks_data = np.concatenate([result_yolo26x_masks_data, result_yolov8x_masks_data], axis=0)
    scores_combined = np.concatenate([scores_yolo26x, scores_yolov8x], axis=0)

    result_nms_masks_data, scores_nms = non_max_suppression_masks(result_combined_masks_data, scores_combined)
    yolo_binary = accumulate_masks(result_nms_masks_data, scores_nms)
    yolo_binary = (yolo_binary > 0).astype(np.uint8)


    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    preds_uneteffnetb5 = predict_with_tta(model_uneteffnetb5, img, CONFIG_UNetEffNetB5.IMG_SIZE, DEVICE)
    preds_uneteffnetb4 = predict_with_tta(model_uneteffnetb4, img, CONFIG_UNetEffNetB4.IMG_SIZE, DEVICE)
    preds_uneteffnetb3 = predict_with_tta(model_uneteffnetb3, img, CONFIG_UNetEffNetB3.IMG_SIZE, DEVICE)
    preds_unetppeffnetb2 = predict_with_tta(model_unetppeffnetb2, img, CONFIG_UNetPPEffNetB2.IMG_SIZE, DEVICE)
    preds_combined = (preds_uneteffnetb5 + preds_uneteffnetb4 + preds_uneteffnetb3 + preds_unetppeffnetb2) / 4.0
    
    smp_binary = (preds_combined > 0.45).astype(np.uint8)
    if smp_binary.shape != (2048, 2048):
        smp_binary = cv2.resize(smp_binary, (2048, 2048), interpolation=cv2.INTER_NEAREST)


    combined_binary = yolo_binary + smp_binary
    combined_binary = (combined_binary > 1).astype(np.uint8)
    refined_binary = refine_mask(combined_binary)
    n_lbl, labels, stats, _ = cv2.connectedComponentsWithStats(refined_binary, connectivity=8)
    
    filament_count = 0
    for lbl in range(1, n_lbl):
        if stats[lbl, cv2.CC_STAT_AREA] < 200: continue
        filament_mask = (labels == lbl).astype(np.uint8)
        filament_count += 1            
        submission_rows.append({
            'filament_id': f"{file_stem}_{filament_count}",
            'segmentation_rle': mask_to_rle(filament_mask),
        })
    '''
    final_masks = remove_instance_overlaps(result_nms_masks_data, scores_nms)
    
    for i, m in enumerate(final_masks):
        m_res = cv2.resize(m.astype(np.uint8), (2048, 2048), interpolation=cv2.INTER_NEAREST)
        if m_res.sum() < 5: continue
        submission_rows.append({
            "filament_id": f"{file_stem}_{i+1}",
            "segmentation_rle": mask_to_rle(m_res)
        })
    '''

Inference: 100%|██████████| 180/180 [41:16<00:00, 13.76s/it]


In [6]:
sub_path = "/kaggle/working/submission.csv"
with open(sub_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["filament_id", "segmentation_rle"])
    writer.writeheader()
    writer.writerows(submission_rows)
print(f"Submission saved to {sub_path} ({len(submission_rows)} instances)")

Submission saved to /kaggle/working/submission.csv (1372 instances)
